# ESG Signal Detection in Corporate Earnings Slides — Pipeline Overview

This notebook walks through the full research pipeline end-to-end:

1. **Vision Analysis** — How Gemini 2.5 Flash classifies each slide for ESG/CSR content
2. **Feature Aggregation** — Converting per-slide labels into per-presentation ratios
3. **Financial Merging** — Linking CSR features to stock market reaction data (CAR)
4. **Regression Analysis** — Testing whether ESG signals predict abnormal returns
5. **Fine-tuning Data Prep** — Building a training dataset from labeled slides

> **Note:** This notebook requires the data files (PDFs, CAR CSV, index Excel).
> The analysis scripts in `src/` can be run independently via CLI (see README).

In [ ]:
import os
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from dotenv import load_dotenv

# Add repo root to path
sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

print('Environment loaded.')

---
## Step 1: Vision Analysis

The `analyze_gemini.py` script converts each presentation PDF into slide images, then sends
all slides to **Gemini 2.5 Flash** in a single API call with a structured prompt.

The model classifies each slide on three dimensions:

| Dimension | Labels |
|-----------|--------|
| CSR-related | Yes / No |
| Category | CSR Forward / CSR Summary / Others |
| ESG Domain | Environmental / Social / Governance / Mixed |

**CSR Forward** = future goals and targets (e.g. "Our 2030 net-zero commitment")

**CSR Summary** = past achievements (e.g. "We reduced emissions by 30% in 2023")

Run from CLI:
```bash
python src/pipeline/analyze_gemini.py --year 2018
```

In [ ]:
# Show example input: a raw slide image
fig, ax = plt.subplots(figsize=(12, 7))
img = mpimg.imread('../examples/slides/slide_page_1.jpg')
ax.imshow(img)
ax.axis('off')
ax.set_title('Example Input: Raw Presentation Slide', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Example of what the Gemini API returns for one presentation
example_output = {
    "filename": "1794718817.pdf",
    "analysis": {
        "overall_summary": {
            "primary_csr_focus": "Environmental",
            "overall_tone": "Mixed",
            "summary_text": "The presentation includes both backward-looking sustainability "
                            "achievements and forward-looking net-zero commitments."
        },
        "slide_analysis": [
            {"slide_number": 1, "csr_related": "No", "breakdown": None,
             "category": "N/A", "reasoning": "Title slide, no ESG content."},
            {"slide_number": 2, "csr_related": "Yes",
             "breakdown": {"is_environmental": "Yes", "is_social": "No", "is_governance": "No"},
             "category": "CSR Summary",
             "reasoning": "Solar panel imagery with 2023 energy savings reported."},
            {"slide_number": 3, "csr_related": "Yes",
             "breakdown": {"is_environmental": "Yes", "is_social": "Yes", "is_governance": "No"},
             "category": "CSR Forward",
             "reasoning": "Net-zero 2035 target and community investment pledge."},
        ]
    }
}

print(json.dumps(example_output, indent=2))

---
## Step 2: Feature Aggregation

The `merge_results.py` script reads the JSONL output and aggregates slide-level labels
into per-presentation feature ratios:

- **csr_ratio**: fraction of slides with any CSR content
- **env_ratio**: fraction with Environmental content
- **soc_ratio**: fraction with Social content
- **gov_ratio**: fraction with Governance content
- **csr_forward_ratio**: fraction of CSR slides that are forward-looking
- **csr_summary_ratio**: fraction of CSR slides that are backward-looking

In [ ]:
from src.analysis.merge_results import analys_results

# Replace with actual path to your Gemini results file:
# df = analys_results('/results/final_csr_analysis_2018.json')

# For demonstration, build a small synthetic dataset
import numpy as np
np.random.seed(42)
n = 100

demo_df = pd.DataFrame({
    'filename': [f'ppt_{i}.pdf' for i in range(n)],
    'total_slides': np.random.randint(20, 150, n),
    'csr_ratio': np.random.beta(2, 5, n),
    'env_ratio': np.random.beta(1.5, 6, n),
    'soc_ratio': np.random.beta(1.5, 6, n),
    'gov_ratio': np.random.beta(1, 8, n),
    'csr_forward_ratio': np.random.beta(2, 4, n),
})

print(f'Feature table shape: {demo_df.shape}')
demo_df.head()

---
## Step 3: Statistical Analysis

We regress Cumulative Abnormal Returns (CAR) on CSR feature ratios, controlling for
firm fundamentals (P/E, leverage, profitability, etc.) and year/industry fixed effects.

We use **HC3 heteroskedasticity-robust standard errors** to account for variance in
the cross-sectional data.

In [ ]:
from src.analysis.regression import run_ols_regression_robust

# Simulate CAR data with a small positive relationship to csr_ratio
demo_df['car_3factor'] = (
    0.02 * demo_df['csr_ratio']
    + 0.01 * demo_df['env_ratio']
    + np.random.normal(0, 0.05, n)
)

model = run_ols_regression_robust(
    demo_df,
    x_var='csr_ratio',
    y_var='car_3factor',
    verbose=True
)

---
## Step 4: Visualizing Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Correlation heatmap
img1 = mpimg.imread('../examples/results/pearson_correlation.png')
axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Pearson Correlation: CSR Features vs. Returns', fontsize=12)

# Right: Event study
img2 = mpimg.imread('../examples/results/event_study_car.png')
axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Event Study: CAR Around Earnings Call Dates', fontsize=12)

plt.tight_layout()
plt.show()

---
## Step 5: Fine-tuning Data Preparation

Once we have labeled slides from the Gemini pipeline, we can use `prepare_finetune_dataset.py`
to package them as training data for fine-tuning a smaller, cheaper VLM.

The goal: train a model that classifies ESG slides as accurately as Gemini 2.5 Flash,
but at a fraction of the inference cost.

```bash
# Build HuggingFace-format dataset
python src/pipeline/prepare_finetune_dataset.py \
    --results-jsonl /results/final_csr_analysis_2018.json \
    --images-dir /data/CSR/images \
    --output-dir ./finetune_data \
    --format huggingface

# Fine-tune Qwen2-VL-2B with LoRA (GPU required)
python src/pipeline/finetune_vlm.py \
    --backend huggingface \
    --base-model Qwen/Qwen2-VL-2B-Instruct \
    --train-jsonl finetune_data/train.jsonl \
    --val-jsonl finetune_data/val.jsonl \
    --epochs 3
```

In [ ]:
# Visualize the class distribution of a hypothetical labeled dataset
labels = ['CSR Forward', 'CSR Summary', 'Others', 'Not CSR']
counts = [1800, 2400, 900, 8900]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, counts, color=['#2196F3', '#4CAF50', '#FF9800', '#9E9E9E'])
ax.set_ylabel('Number of Slides')
ax.set_title('Example Class Distribution in Fine-tuning Dataset')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

---
## Summary

This pipeline demonstrates how to:

1. Apply **multimodal AI** (vision models) to extract structured signals from unstructured business data
2. Build a **research-grade dataset** by combining AI-generated labels with financial market data
3. Use **econometric methods** (event study, OLS with robust SEs) to test financial hypotheses
4. **Distill a large model's knowledge** into a smaller fine-tuned model for cost-efficient inference

See the README for full setup instructions and CLI usage.